<a href="https://colab.research.google.com/github/Geethanjali-345/19032024/blob/geethanjali/TestcaseGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **1.Preprocessing**

In [1]:
#importing
import re
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


#nltk.download('punkt')
#nltk.download('stopwords')
#nltk.download('wordnet')

#file loading
file_path = '/content/nfr.txt'
with open(file_path, "r") as file:
    content = file.read()

# Step 1: Data Cleaning
content_clean = re.sub(r'\n+', ' ', content)
content_clean = re.sub(r'\s+', ' ', content)
content_clean = re.sub(r'[^a-zA-Z\s]', '', content_clean)
content_clean = content_clean.lower()

# Step 2: Sentence Tokenization
sentences = sent_tokenize(content_clean)

# Step 3: Word Tokenization
tokens = [word_tokenize(sentence) for sentence in sentences]

# Step 4: Stop Words Removal
stop_words = set(stopwords.words('english'))
filtered_tokens = [[word for word in sentence if word not in stop_words] for sentence in tokens]

# Step 5: Lemmatization
lemmatizer = WordNetLemmatizer()
lemmatized_tokens = [[lemmatizer.lemmatize(word) for word in sentence] for sentence in filtered_tokens]

# Step 6: Rejoin Tokens for Categorization
preprocessed_text = [' '.join(sentence) for sentence in lemmatized_tokens]



### **2.Categorization**

##### So that we can easily monitor the different categories of requirements easily :)

In [2]:
from collections import defaultdict
requirements = defaultdict(list)


for line in content.splitlines():
    if line.strip():
        parts = line.split(":", 1)
        if len(parts) == 2:
            prefix = parts[0].strip()
            content = parts[1].strip()

            # Add the content to the appropriate category in the dictionary
            requirements[prefix].append(content)



# number of requirements in each category
for key, reqs in requirements.items():
    print(f"{key}: {len(reqs)} requirements")



PE: 54 requirements
LF: 38 requirements
US: 67 requirements
A: 21 requirements
SE: 66 requirements
F: 255 requirements
L: 13 requirements
O: 62 requirements
SC: 21 requirements
FT: 10 requirements
MN: 17 requirements
PO: 1 requirements


In [3]:
category_key = "PO"  # Replace with the desired category key
if category_key in requirements:
    category_requirements = requirements[category_key]

    print(f"Requirements for category '{category_key}':")
    for req in category_requirements:
        print(f"- {req}")

Requirements for category 'PO':
- The product is expected to run on Windows CE and Palm operating systems.


In [4]:
#pip install spacy

### **3.Custom NER**

In [5]:
import spacy

In [18]:
#!python -m spacy download en_core_web_sm


In [11]:
import json
from spacy.matcher import PhraseMatcher
nlp = spacy.load("en_core_web_sm")

# Define custom NER function
def custom_ner(doc, custom_entities):
    actors = []
    actions = []
    conditions = []

    # Creating phrase matchers for multi-token entities
    matcher = PhraseMatcher(nlp.vocab)

    # Ading custom entities to matchers
    actor_patterns = [nlp.make_doc(actor.lower()) for actor in custom_entities['Actor']]
    action_patterns = [nlp.make_doc(action.lower()) for action in custom_entities['Action']]
    condition_patterns = [nlp.make_doc(condition.lower()) for condition in custom_entities['Condition']]

    matcher.add("ACTOR", None, *actor_patterns)
    matcher.add("ACTION", None, *action_patterns)
    matcher.add("CONDITION", None, *condition_patterns)

    # Finding matches for actors, actions, and conditions
    matches = matcher(doc)

    for match_id, start, end in matches:
        span = doc[start:end].text
        label = nlp.vocab.strings[match_id]

        if label == "ACTOR":
            actors.append(span)
        elif label == "ACTION":
            actions.append(span)
        elif label == "CONDITION":
            conditions.append(span)

    return actors, actions, conditions, []

# Function to process requirements
def process_requirements(requirements, custom_entities, replace_terms=None):
    results = []
    for requirement in requirements:
        if replace_terms:
            for old_term, new_term in replace_terms.items():
                requirement = requirement.replace(old_term, new_term)

        # Tokenize and analyze the text with spaCy
        doc = nlp(requirement)

        # Extract custom entities
        actors, actions, conditions, _ = custom_ner(doc, custom_entities)

        # For now, use the entire requirement as the outcome
        outcomes = [requirement]

        results.append({
            "Requirement": requirement,
            "Actors": actors,
            "Actions": actions,
            "Conditions": conditions,
            "Outcomes": outcomes
        })
    return results

# Loading custom entities from JSON file and convert to dictionary
with open('/content/custom_entities.json', 'r') as f:
    custom_entities = json.load(f)

#loading requiremnts
requirements_po = requirements['PO']
requirements_ft = requirements['FT']
requirements_l = requirements['L']
requirements_mn = requirements['MN']
requirements_a = requirements['A']
requirements_sc = requirements['SC']
requirements_lf = requirements['LF']
requirements_pe = requirements['PE']
requirements_se = requirements['SE']
requirements_o = requirements['O']
requirements_us = requirements['US']
requirements_f = requirements['F']

# Process each set of requirements
results_po = process_requirements(requirements_po, custom_entities['PO'])
results_ft = process_requirements(requirements_ft, custom_entities['FT'], replace_terms={"shall": "should"})
results_l = process_requirements(requirements_l, custom_entities['L'], replace_terms={"shall": "should", "will": "should", "must": "should"})
results_mn = process_requirements(requirements_mn, custom_entities['MN'], replace_terms={"shall": "should", "will": "should", "must": "should"})
results_a = process_requirements(requirements_a, custom_entities['A'], replace_terms={"shall": "should", "will": "should", "must": "should"})
results_sc = process_requirements(requirements_sc, custom_entities['SC'], replace_terms={"shall": "should", "will": "should", "must": "should"})
results_lf = process_requirements(requirements_lf, custom_entities['LF'], replace_terms={"shall": "should", "will": "should", "must": "should"})
results_pe = process_requirements(requirements_pe, custom_entities['PE'], replace_terms={"shall": "should", "will": "should", "must": "should"})
results_se = process_requirements(requirements_se, custom_entities['SE'], replace_terms={"shall": "should", "will": "should", "must": "should"})
results_o = process_requirements(requirements_o, custom_entities['O'], replace_terms={"shall": "should", "will": "should", "must": "should"})
results_us = process_requirements(requirements_us, custom_entities['US'], replace_terms={"shall": "should", "will": "should", "must": "should"})
results_f = process_requirements(requirements_f, custom_entities['F'], replace_terms={"shall": "should", "will": "should", "must": "should"})

# Combine all results
output = {
    "PO": results_po,
    "FT": results_ft,
    "L": results_l,
    "MN": results_mn,
    "A": results_a,
    "SC": results_sc,
    "LF": results_lf,
    "PE": results_pe,
    "SE": results_se,
    "O": results_o,
    "US": results_us,
    "F": results_f
}

# Print or save the output
#print(json.dumps(output, indent=2))


### **4.Rule based approach for Test case generation**

In [17]:
import json

def parse_requirements(requirements_data):
    if isinstance(requirements_data, str):
        try:
            return json.loads(requirements_data)
        except json.JSONDecodeError:
            print(f"Error decoding JSON: {requirements_data}")
            return {}
    return requirements_data

def rule_based_test_case_generator(parsed_requirements):
    test_cases = []

    for category, requirements in parsed_requirements.items():
        for req in requirements:
            requirement = parse_requirements(req)

            if isinstance(requirement, dict):
                actions = requirement.get('Actions', [])
                actors = requirement.get('Actors', [])
                conditions = requirement.get('Conditions', ['No specific conditions identified'])
                outcomes = requirement.get('Outcomes', [])

                # Generate multiple test cases by combining actions, actors, and conditions
                for action in actions:
                    for actor in actors:
                        for condition in conditions:
                            for outcome in outcomes:
                                test_case = {
                                    'Test Case Title': generate_test_case_title(requirement, action, condition),
                                    'Test Steps': generate_test_steps(requirement, action, condition),
                                    'Preconditions': generate_preconditions(requirement, condition),
                                    'Expected Result': generate_expected_result(requirement)
                                }
                                test_cases.append(test_case)
            else:
                print(f"Skipping invalid requirement format: {requirement}")

    return test_cases

def generate_test_case_title(requirement, action, condition):
    actor = requirement.get('Actors', [''])[0]
    title = f"Verify {action} by {actor}"
    if condition and condition != 'No specific conditions identified':
        title += f" with {condition}"
    return title

def generate_test_steps(requirement, action, condition):
    steps = []
    steps.append(f"Step 1: Ensure the following conditions are met: {condition}.")
    steps.append(f"Step 2: Perform the action '{action}'.")
    steps.append(f"Step 3: Verify the outcome matches the expected result: '{requirement.get('Outcomes', [''])[0]}'")
    return '\n'.join(steps)

def generate_preconditions(requirement, condition):
    return condition

def generate_expected_result(requirement):
    return requirement.get('Outcomes', ['No expected result provided'])[0]


with open('/content/parsed_requirements.json') as f:
    parsed_requirements = json.load(f)

# Generate test cases using rule-based approach
test_cases = rule_based_test_case_generator(parsed_requirements)

# Function to print the first 100 test cases and total number of test cases
def print_first_100_test_cases(test_cases):
    total_cases = len(test_cases)
    print(f"Total number of test cases: {total_cases}\n")

    for i, case in enumerate(test_cases[:100], 1):  # Print only the first 100 test cases
        print(f"Test Case {i}")
        print("-" * 80)
        print(f"Title: {case['Test Case Title']}")
        print("Steps:")
        print(f"{case['Test Steps']}")
        print(f"Preconditions: {case['Preconditions']}")
        print(f"Expected Result: {case['Expected Result']}")
        print("\n" + "=" * 80 + "\n")

    if total_cases > 100:
        print(f"Note: Only the first 100 of {total_cases} test cases are displayed.")

#print_first_100_test_cases(test_cases)
